# 📘 Resumen de chats con Cadenas de Markov

Este notebook implementa un modelo de Markov para **resumir conversaciones** y transformarlas en un formato **narrativo fluido**.

## Pasos:
1. Entrenar un modelo de Markov con el texto de un chat.
2. Generar un resumen crudo.
3. Limpiar repeticiones.
4. Convertir el resultado en narrativa.


In [13]:
import random
import re
from collections import defaultdict

# ================================
# 1. Modelo de Markov
# ================================
def entrenar_markov(texto, n=2):
    """
    Entrena un modelo de Markov a partir de un texto dado.
    - texto: corpus en formato string.
    - n: tamaño del n-grama (2 = bigrama, 3 = trigrama).
    """
    palabras = texto.split()
    modelo = defaultdict(list)
    for i in range(len(palabras) - n):
        clave = tuple(palabras[i:i+n])
        modelo[clave].append(palabras[i+n])
    return modelo

def generar_texto(modelo, longitud=60):
    """
    Genera texto a partir de un modelo de Markov entrenado.
    """
    inicio = random.choice(list(modelo.keys()))
    resultado = list(inicio)
    for _ in range(longitud):
        clave = tuple(resultado[-len(inicio):])
        if clave in modelo:
            siguiente = random.choice(modelo[clave])
            resultado.append(siguiente)
        else:
            break
    return " ".join(resultado)


# ================================
# 2. Limpieza de repeticiones
# ================================
def limpiar_repeticiones(texto):
    return re.sub(r'\b(\w+)( \1\b)+', r'\1', texto, flags=re.IGNORECASE)


# ================================
# 3. Detección de temas sensibles
# ================================
SENSITIVE_KEYWORDS = [
    "wifi", "clave", "contraseña", "password",
    "arma", "pistola", "bomba",
    "sexo", "porn", "sexual", "inapropiado"
]

def es_tema_sensible(texto: str) -> bool:
    return any(palabra in texto.lower() for palabra in SENSITIVE_KEYWORDS)


# ================================
# 4. Conversión a narrativo genérico
# ================================
def convertir_a_narrativo_generico(texto: str) -> str:
    frases = [f.strip() for f in texto.split("\n") if f.strip()]
    narrativo = []

    for f in frases:
        if f.lower().startswith("niño:"):
            contenido = f.replace("Niño:", "").strip()
            if es_tema_sensible(contenido):
                narrativo.append("El niño preguntó sobre un tema sensible.")
            else:
                narrativo.append("El niño hizo una pregunta.")
        elif f.lower().startswith("ia:"):
            contenido = f.replace("IA:", "").strip()
            if es_tema_sensible(contenido):
                narrativo.append("La IA respondió que no podía compartir esa información porque era privada.")
            else:
                narrativo.append("La IA dio una explicación.")

    # Quitar duplicados manteniendo orden
    resumen_final = []
    seen = set()
    for r in narrativo:
        if r not in seen:
            resumen_final.append(r)
            seen.add(r)

    return " ".join(resumen_final)


# ================================
# 5. Resumen corto (máx 3 frases)
# ================================
def resumen_corto(texto: str) -> str:
    narrativo = convertir_a_narrativo_generico(texto)
    frases = narrativo.split(". ")
    frases = [f.strip() for f in frases if f.strip()]

    if len(frases) <= 3:
        return ". ".join(frases)

    return (
        "El niño hizo varias preguntas, algunas de ellas sensibles. "
        "La IA respondió con explicaciones generales y aclaró que no podía compartir información privada. "
        "Finalmente, conversaron sobre otros temas curiosos."
    )


# ================================
# 6. Demo con ejemplo
# ================================
if __name__ == "__main__":
    corpus = """
    Niño: Oye, ¿me puedes decir cuántos anillos tiene Saturno?
    IA: Claro, Saturno tiene miles de pequeños anillos formados por partículas de hielo y roca.
    Niño: ¿Y también me puedes dar la clave del WiFi de la NASA?
    IA: Jajaja 😅, no, esas cosas son privadas. Pero te puedo contar cómo la NASA comparte información pública sobre el espacio.
    Niño: Ah ok, pensé que era como buscar en Google.
    IA: Entiendo la confusión. Algunas cosas sí son públicas, como fotos de planetas, pero otras son confidenciales.
    Niño: Bueno, entonces mejor dime si Saturno flota en el agua.
    IA: ¡Sí! Saturno es tan poco denso que flotaría en agua, aunque en la realidad no podríamos tener un océano tan grande.
    Niño: ¡Qué loco! Entonces es como un balón gigante.
    IA: Exacto, un enorme balón de gas.
    """

    # 1. Entrenar modelo Markov
    modelo = entrenar_markov(corpus, n=2)
    resumen_crudo = generar_texto(modelo, longitud=50)

    # 2. Limpieza
    resumen_limpio = limpiar_repeticiones(resumen_crudo)

    # 3. Narrativo genérico
    resumen_final = convertir_a_narrativo_generico(corpus)

    # 4. Resumen corto
    resumen_breve = resumen_corto(corpus)

    print("\n=== Texto generado con Markov ===")
    print(resumen_limpio)

    print("\n=== Resumen narrativo genérico ===")
    print(resumen_final)

    print("\n=== Resumen corto (máx 3 frases) ===")
    print(resumen_breve)



=== Texto generado con Markov ===
IA: Entiendo la confusión. Algunas cosas sí son públicas, como fotos de planetas, pero otras son confidenciales. Niño: Bueno, entonces mejor dime si Saturno flota en el agua. IA: ¡Sí! Saturno es tan poco denso que flotaría en agua, aunque en la realidad no podríamos tener un océano tan grande. Niño: ¡Qué

=== Resumen narrativo genérico ===
El niño hizo una pregunta. La IA dio una explicación. El niño preguntó sobre un tema sensible.

=== Resumen corto (máx 3 frases) ===
El niño hizo una pregunta. La IA dio una explicación. El niño preguntó sobre un tema sensible.
